# Bayesian Networks

In [1]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

# 1. Define a estrutura do grafo (quem aponta para quem)
model = DiscreteBayesianNetwork([
    ("rain", "maintenance"),
    ("rain", "train"),
    ("maintenance", "train"),
    ("train", "appointment")
])

# 2. Define as tabelas de probabilidade (CPDs)

# Rain (sem pais)
cpd_rain = TabularCPD(
    variable="rain",
    variable_card=3,
    values=[[0.7], [0.2], [0.1]],
    state_names={"rain": ["none", "light", "heavy"]}
)

# Maintenance (depende de rain)
cpd_maintenance = TabularCPD(
    variable="maintenance",
    variable_card=2,
    values=[
        [0.4, 0.2, 0.1],  # yes
        [0.6, 0.8, 0.9]   # no
    ],
    evidence=["rain"],
    evidence_card=[3],
    state_names={
        "maintenance": ["yes", "no"],
        "rain": ["none", "light", "heavy"]
    }
)

# Train (depende de rain e maintenance)
cpd_train = TabularCPD(
    variable="train",
    variable_card=2,
    values=[
        # on time
        [0.8, 0.9, 0.6, 0.7, 0.4, 0.5],
        # delayed
        [0.2, 0.1, 0.4, 0.3, 0.6, 0.5]
    ],
    evidence=["rain", "maintenance"],
    evidence_card=[3, 2],
    state_names={
        "train": ["on time", "delayed"],
        "rain": ["none", "light", "heavy"],
        "maintenance": ["yes", "no"]
    }
)

# Appointment (depende de train)
cpd_appointment = TabularCPD(
    variable="appointment",
    variable_card=2,
    values=[
        [0.9, 0.6],  # attend
        [0.1, 0.4]   # miss
    ],
    evidence=["train"],
    evidence_card=[2],
    state_names={
        "appointment": ["attend", "miss"],
        "train": ["on time", "delayed"]
    }
)

# 3. Adiciona as tabelas ao modelo, valida e prepara a inferência
model.add_cpds(cpd_rain, cpd_maintenance, cpd_train, cpd_appointment)
assert model.check_model()

infer = VariableElimination(model)

/home/cauan/AI eng/projetos/introduction_to_artificial_inteligence_with_python/.venv/lib/python3.13/site-packages/pgmpy/estimators/__init__.py:4: FutureWarning: `pgmpy.estimators.StructureScore` is deprecated and will be removed in v1.3.0. Use `pgmpy.structure_score` instead.
  from .StructureScore import (


In [2]:
# Calcula a probabilidade conjunta P(rain="none", maintenance="no", train="on time", appointment="attend")
joint_dist = infer.query(variables=["rain", "maintenance", "train", "appointment"])

probability = joint_dist.get_value(
    rain="none",
    maintenance="no",
    train="on time",
    appointment="attend"
)

print(f"Probabilidade conjunta: {probability:.4f}")

Probabilidade conjunta: 0.3402


In [5]:
evidence = {"maintenance": "yes"}

for node in model.nodes():
    if node in evidence:
        print(f"{node}: {evidence[node]}")
    else:
        print(node)
        dist = infer.query(variables=[node], evidence=evidence)
        for state, prob in zip(dist.state_names[node], dist.values):
            print(f"    {state}: {prob:.4f}")

rain
    none: 0.8485
    light: 0.1212
    heavy: 0.0303
maintenance: yes
train
    on time: 0.7636
    delayed: 0.2364
appointment
    attend: 0.8291
    miss: 0.1709


### Sampling

In [23]:
from pgmpy.sampling import BayesianModelSampling
from pgmpy.factors.discrete import State

sampler = BayesianModelSampling(model)

N = 10000

# 1. Define a evidência fixa: rain = "heavy"
evidence = [State(var="rain", state="heavy")]

# 2. Gera N amostras com seus respectivos pesos (_weight)
samples = sampler.likelihood_weighted_sample(
    evidence=evidence,
    size=N,
    show_progress=False
)

# 3. Soma os pesos agrupados pelo resultado de "appointment" e normaliza
weighted_counts = samples.groupby("appointment")["_weight"].sum()
probabilities = weighted_counts / weighted_counts.sum()

print("Soma dos pesos por estado:")
print(weighted_counts)
print("\nProbabilidades estimadas:")
print(probabilities)

Soma dos pesos por estado:
appointment
attend    747.1
miss      252.9
Name: _weight, dtype: float64

Probabilidades estimadas:
appointment
attend    0.7471
miss      0.2529
Name: _weight, dtype: float64


# Markov Models

In [26]:
import random
from pgmpy.models import MarkovChain
from pgmpy.factors.discrete import State

# 1. Cria a cadeia de Markov passando a variável e sua cardinalidade
model = MarkovChain(["weather"], [2])

# 2. Define o modelo de transição usando os índices (0 = sun, 1 = rain)
# ou passando o state_names caso sua versão suporte
model.add_transition_model(
    "weather",
    {
        0: {0: 0.8, 1: 0.2},
        1: {0: 0.3, 1: 0.7}
    }
)

# 3. Sorteia o estado inicial (50/50) e gera 50 passos
state_map = {0: "sun", 1: "rain"}
initial_state = random.choices([0, 1], weights=[0.5, 0.5])[0]

samples = model.sample(start_state=[State("weather", initial_state)], size=50)
print([state_map[s] for s in samples["weather"]])

['rain', 'rain', 'sun', 'sun', 'sun', 'sun', 'rain', 'sun', 'sun', 'sun', 'sun', 'sun', 'sun', 'sun', 'sun', 'sun', 'rain', 'rain', 'sun', 'sun', 'sun', 'rain', 'rain', 'sun', 'sun', 'rain', 'rain', 'rain', 'sun', 'sun', 'rain', 'sun', 'sun', 'sun', 'sun', 'rain', 'rain', 'rain', 'sun', 'sun', 'sun', 'sun', 'rain', 'sun', 'sun', 'sun', 'sun', 'sun', 'sun', 'sun']


## Hidden Markov Models

In [28]:
import numpy as np

# Nomes dos estados ocultos e observações
state_names = ["sun", "rain"]
obs_names = ["umbrella", "no umbrella"]

# Probabilidades iniciais (starts)
starts = np.array([0.5, 0.5])

# Matriz de transição (transitions)
# Linha 0 (sun) -> [sun: 0.8, rain: 0.2]
# Linha 1 (rain) -> [sun: 0.3, rain: 0.7]
transitions = np.array([
    [0.8, 0.2],
    [0.3, 0.7]
])

# Matriz de emissão / observação (states)
# Linha 0 (sun) -> [umbrella: 0.2, no umbrella: 0.8]
# Linha 1 (rain) -> [umbrella: 0.9, no umbrella: 0.1]
emissions = np.array([
    [0.2, 0.8],
    [0.9, 0.1]
])

def viterbi_predict(observations):
    obs_idx = [obs_names.index(o) for o in observations]
    n_states = len(state_names)
    n_obs = len(obs_idx)

    # Tabelas de probabilidade máxima e ponteiros de caminho
    dp = np.zeros((n_states, n_obs))
    backpointer = np.zeros((n_states, n_obs), dtype=int)

    # Passo inicial (t = 0)
    dp[:, 0] = starts * emissions[:, obs_idx[0]]

    # Passos seguintes (t = 1 ... n_obs - 1)
    for t in range(1, n_obs):
        for s in range(n_states):
            probs = dp[:, t - 1] * transitions[:, s] * emissions[s, obs_idx[t]]
            backpointer[s, t] = np.argmax(probs)
            dp[s, t] = np.max(probs)

    # Reconstrução da sequência mais provável (backtracking)
    best_path = [np.argmax(dp[:, -1])]
    for t in range(n_obs - 1, 0, -1):
        best_path.append(backpointer[best_path[-1], t])
    best_path.reverse()

    return best_path

In [29]:
# Observed data
observations = [
    "umbrella",
    "umbrella",
    "no umbrella",
    "umbrella",
    "umbrella",
    "umbrella",
    "umbrella",
    "no umbrella",
    "no umbrella"
]

# Predict underlying states
predictions = viterbi_predict(observations)
for prediction in predictions:
    print(state_names[prediction])

rain
rain
rain
rain
rain
rain
rain
sun
sun
